In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import os
import numpy
import sklearn
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import sklearn.model_selection as skms

import lightgbm
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
dfTrain = pd.read_csv("./train.csv")

In [4]:
dfTrain.head()

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [5]:
def preProc (d):
    if "id" in list(d):
        d = d.drop("id",axis = 1)
    mapping = {True:1,False:0}
    d["road_signs_present"] = d["road_signs_present"].map(mapping)
    d["public_road"] = d["public_road"].map(mapping)
    d["holiday"] = d["holiday"].map(mapping)
    d["school_season"] = d["school_season"].map(mapping)
    return(d)



xTrain = preProc(dfTrain).drop("accident_risk", axis = 1)
yTrain = dfTrain["accident_risk"]

#xTrain, xTrainTest, yTrain, yTrainTest = skms.train_test_split(xTrain, yTrain, test_size = 0.05, train_size = 0.95, random_state = 4000)


In [6]:
dfTest = pd.read_csv("./test.csv")
xTest = preProc(dfTest)

In [7]:

categorical_cols = ['road_type', 'lighting', 'weather', 'time_of_day']
numeric_cols = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents']


preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
], remainder='passthrough', force_int_remainder_cols=False)  

In [8]:
from sklearn.pipeline import Pipeline
#from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor


model = Pipeline([
    ('preprocess', preprocessor),
    ('regressor', LGBMRegressor(
        n_estimators=200,
        max_depth=20,
        random_state=4000,
        n_jobs=-1
    ))
])


In [9]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_distributions = {
    'regressor__n_estimators': randint(100, 600),
    'regressor__max_depth': randint(10, 40),
    'regressor__min_samples_split': randint(2, 30),
    'regressor__min_samples_leaf': randint(1, 30),
    'regressor__max_features': ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=20,
    cv=7,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    random_state=4000,
    verbose=1
)

random_search.fit(xTrain, yTrain)
best_model = random_search.best_estimator_
print(random_search.best_params_)


Fitting 7 folds for each of 200 candidates, totalling 1400 fits
[LightGBM] [Warning] min_data_in_leaf is set with min_child_samples=20, will be overridden by min_samples_leaf=17. Current value: min_data_in_leaf=17
[LightGBM] [Warning] Unknown parameter: min_samples_split
[LightGBM] [Warning] min_data_in_leaf is set with min_child_samples=20, will be overridden by min_samples_leaf=17. Current value: min_data_in_leaf=17
[LightGBM] [Warning] Unknown parameter: min_samples_split
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005590 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 209
[LightGBM] [Info] Number of data points in the train set: 517754, number of used features: 20
[LightGBM] [Info] Start training from score 0.352377
{'regressor__max_depth': 11, 'regressor__max_features': None, 'regressor__min_samples_leaf': 17, 'regressor__min_s

In [10]:
#model.fit(xTrain, yTrain)

In [11]:
model = best_model

In [12]:
#yTrainTestPred = best_model.predict(xTrainTest)

In [13]:
'''from sklearn.metrics import mean_squared_error
mse = mean_squared_error(yTrainTest, yTrainTestPred)
rmse = np.sqrt(mse)

print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)'''

'from sklearn.metrics import mean_squared_error\nmse = mean_squared_error(yTrainTest, yTrainTestPred)\nrmse = np.sqrt(mse)\n\nprint("Mean Squared Error:", mse)\nprint("Root Mean Squared Error:", rmse)'

In [14]:
yTest = model.predict(xTest)

[LightGBM] [Warning] min_data_in_leaf is set with min_child_samples=20, will be overridden by min_samples_leaf=17. Current value: min_data_in_leaf=17
[LightGBM] [Warning] Unknown parameter: min_samples_split


In [15]:
dfTrain.head()

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [16]:
a = yTest.reshape(len(yTest), 1)
b = numpy.asarray(dfTest['id']).reshape(len(yTest), 1)
numpy.savetxt("yTest.csv", numpy.concatenate((b,a),axis = 1),header = 'id,accident_risk', delimiter=",", comments='')

In [17]:
yTestCsv = pd.read_csv("./yTest.csv")
yTestCsv

,id,accident_risk
0,517754.0,0.290354
1,517755.0,0.123946
2,517756.0,0.186001
3,517757.0,0.310833
4,517758.0,0.406203
...,...,...
172580,690334.0,0.111485
172581,690335.0,0.514712
172582,690336.0,0.249426
172583,690337.0,0.127974


In [18]:
ySamp = pd.read_csv("./sample_submission.csv")
ySamp

,id,accident_risk
0,517754,0.352
1,517755,0.352
2,517756,0.352
3,517757,0.352
4,517758,0.352
...,...,...
172580,690334,0.352
172581,690335,0.352
172582,690336,0.352
172583,690337,0.352


In [19]:
list(dfTest)

['id',
 'road_type',
 'num_lanes',
 'curvature',
 'speed_limit',
 'lighting',
 'weather',
 'road_signs_present',
 'public_road',
 'time_of_day',
 'holiday',
 'school_season',
 'num_reported_accidents']